# Graph Creation Deep Dive

In Neural‑LAM, the atmosphere is represented as a graph. This tutorial provides a deep dive into the graph theory, architecture, and optimization strategies used to enable efficient weather prediction with Graph Neural Networks (GNNs).

## 1. GNN Fundamentals for Weather Prediction

Traditional Numerical Weather Prediction (NWP) models use fixed grids (finite volume/difference). Neural‑LAM leverages **Interaction Networks (IN)**, a specific type of Message Passing Neural Network (MPNN), to learn these physical interactions. 

### Why Graphs?
- **Irregular Geometry**: Can easily handle the Earth's curvature and varying resolutions.
- **Inductive Bias**: Captures spatial dependencies naturally through edges.
- **Efficiency**: Latent message passing on a sparse graph is often faster than dense grid computations.

### Interaction Net Implementation
In `neural_lam/interaction_net.py`, the core operation follows the architecture from **Battaglia et al. (2016)**. 

The Interaction Network updates edge representations based on the connected nodes and previous edge states, and then updates node representations by aggregating these new edge messages. This process allows the model to learn complex, non-linear physical interactions. For a full mathematical treatment, see **Battaglia et al. (2016)** and the **Neural-LAM paper (Oskarsson et al. 2023)**.

The **Processor** module in models like `GraphLAM` stacks multiple `InteractionNet` layers to propagate information across the mesh graph.

## 2. Flat vs. Hierarchical Architectures

Choosing the right graph structure impacts both computational cost and the model's ability to capture long-range weather patterns.

### Comparison

| Feature | Flat (Multi-scale) | Hierarchical (HiLAM) |
| :--- | :--- | :--- |
| **Structure** | All mesh nodes in one layer. | Tree-like levels with up/down edges. |
| **Global Flow** | Slower (requires many m2m steps). | Fast (logarithmic scaling via hierarchy). |
| **Memory** | Lower (fewer edge types). | Higher (additional inter-level edges). |
| **Best For** | High-res local features. | Large domains / Global interaction. |

## 3. Model Architecture Comparison

Neural-LAM provides three model architectures that work with different graph structures. Each has different trade-offs in terms of computational efficiency, memory usage, and ability to capture long-range dependencies.

### GraphLAM (Flat/Multi-scale)

**File:** `neural_lam/models/graph_lam.py`

GraphLAM uses a flat mesh structure where all mesh nodes exist at a single level. Message passing occurs horizontally through mesh-to-mesh (M2M) edges.

**Characteristics:**
- Single-level mesh with uniform or multi-scale connectivity
- Simpler architecture with fewer edge types
- Lower memory footprint
- Information propagates through multiple M2M steps

**Best for:**
- High-resolution local features
- Smaller domains where long-range interactions are less critical
- When memory is constrained
- Baseline experiments and rapid prototyping

### HiLAM (Hierarchical Sequential)

**File:** `neural_lam/models/hi_lam.py`

HiLAM uses a hierarchical mesh with multiple levels. Message passing proceeds **sequentially**: first down through the hierarchy (fine to coarse), then up (coarse to fine). At each level, same-level message passing is interleaved with inter-level communication.

**Characteristics:**
- Multi-level mesh hierarchy (tree-like structure)
- Sequential down-then-up message passing
- Alternates between inter-level and intra-level processing
- Logarithmic scaling for long-range information flow

**Processing Order:**
1. Process finest level (L)
2. Down to level L-1, process same-level edges
3. Continue down to coarsest level (0)
4. Process coarsest level
5. Up to level 1, process same-level edges
6. Continue up to finest level (L)

**Best for:**
- Large domains requiring long-range interactions
- Global weather modeling
- When you want explicit control over information flow through hierarchy
- Capturing multi-scale atmospheric phenomena

### HiLAM Parallel

**File:** `neural_lam/models/hi_lam_parallel.py`

HiLAM Parallel also uses a hierarchical mesh, but processes **all edges in parallel** rather than sequentially. All mesh-to-mesh, up, and down edges are combined into a single edge set and processed simultaneously.

**Characteristics:**
- Multi-level mesh hierarchy (same as HiLAM)
- Parallel message passing across all edge types
- Simpler implementation than sequential HiLAM
- Single unified message passing step per processor layer

**Processing Approach:**
1. Concatenate all node representations from all levels
2. Concatenate all edge representations (M2M, up, down)
3. Apply message passing to combined graph
4. Split results back into levels

**Best for:**
- Large domains with hierarchical structure
- When you want simpler code than sequential HiLAM
- Potentially faster training due to parallel processing
- When the sequential down-up pattern is not critical

### Architecture Comparison Table

| Feature | GraphLAM | HiLAM (Sequential) | HiLAM Parallel |
| :--- | :--- | :--- | :--- |
| **Graph Structure** | Flat (single level) | Hierarchical (multi-level) | Hierarchical (multi-level) |
| **Message Passing** | Horizontal M2M only | Sequential down-then-up | All edges in parallel |
| **Long-range Flow** | Slow (many steps) | Fast (logarithmic) | Fast (logarithmic) |
| **Memory Usage** | Lower | Higher | Higher |
| **Implementation** | Simple | Complex | Moderate |
| **Edge Types** | M2M, G2M, M2G | M2M, Up, Down, G2M, M2G | M2M, Up, Down, G2M, M2G |
| **Processor Layers** | Single GNN per layer | Multiple GNNs (down/up/same) | Single unified GNN |
| **Best Use Case** | Local high-res | Global multi-scale | Global with simpler code |

### Choosing a Model

**Start with GraphLAM if:**
- You're working with a small domain (< 1000 km)
- You want fast iteration and simple debugging
- Memory is limited

**Use HiLAM (Sequential) if:**
- You need explicit multi-scale processing
- You're modeling large domains or global weather
- You want the architecture from the original Neural-LAM paper

**Use HiLAM Parallel if:**
- You want hierarchical benefits with simpler code
- You're experimenting with hierarchical architectures
- Training speed is important

### Training with Different Architectures

Each model architecture requires a compatible graph structure. Here's how to train with each:

**GraphLAM with Flat Graph:**
```bash
# Create flat graph
python -m neural_lam.create_graph \
  --config_path data/config.yaml \
  --name flat_graph

# Train GraphLAM model
python -m neural_lam.train_model \
  --config_path data/config.yaml \
  --model graph_lam \
  --graph flat_graph
```

**HiLAM with Hierarchical Graph:**
```bash
# Create hierarchical graph
python -m neural_lam.create_graph \
  --config_path data/config.yaml \
  --name hierarchical_graph \
  --hierarchical \
  --levels 3

# Train HiLAM model (sequential)
python -m neural_lam.train_model \
  --config_path data/config.yaml \
  --model hi_lam \
  --graph hierarchical_graph
```

**HiLAM Parallel with Hierarchical Graph:**
```bash
# Use the same hierarchical graph as HiLAM
# Train HiLAM Parallel model
python -m neural_lam.train_model \
  --config_path data/config.yaml \
  --model hi_lam_parallel \
  --graph hierarchical_graph
```

**Important Notes:**
- GraphLAM requires a flat graph (omit --hierarchical flag)
- HiLAM and HiLAM Parallel require a hierarchical graph (include --hierarchical flag)
- Both HiLAM variants can use the same hierarchical graph
- The model architecture is specified with `--model` argument: `graph_lam`, `hi_lam`, or `hi_lam_parallel`
- Use `--config_path` to point to your neural-lam config file (which references the datastore config)
- Use `--name` to specify the graph name, and `--levels` to limit hierarchy depth

## 4. Edge Connectivity & Radius Parameters

The connectivity between the physical grid and the internal mesh is controlled by the **search radius**.

### Grid-to-Mesh (`g2m`) Radius
Internally, `create_graph.py` uses a constant `DM_SCALE = 0.67` multiplied by the mesh node distance ($d_m$).
- **If radius is too small**: Some grid cells are "orphaned" (not connected to any mesh node).
- **If radius is too large**: Excessive edge count increases memory usage and smears local signals.

### Mesh-to-Grid (`m2g`) Connectivity
Unlike `g2m` (ball query), `m2g` uses a nearest-neighbor approach (defaulting to 4 neighbors) to ensure every grid cell receives an updated state.

## 5. Interactive Comparison

Let's see how changing the graph type affects the number of edges and structure.

In [ ]:
from neural_lam.create_graph import create_graph_from_datastore
from neural_lam.config import load_config_and_datastore
from pathlib import Path
import torch

# Load configuration (use relative path from tutorial directory)
config_path = '../data/config.yaml'
config, datastore = load_config_and_datastore(config_path)

# Create three graph variations
variations = [
    {'name': 'flat_l1', 'hierarchical': False, 'levels': 1},
    {'name': 'multiscale_l3', 'hierarchical': False, 'levels': 3},
    {'name': 'hierarchical_l3', 'hierarchical': True, 'levels': 3}
]

for var in variations:
    out_path = Path(f"./temp_graphs/{var['name']}")
    create_graph_from_datastore(
        datastore=datastore,
        output_root_path=str(out_path),
        n_max_levels=var['levels'],
        hierarchical=var['hierarchical']
    )
    
    # Analyze edge counts
    m2m = torch.load(out_path / 'm2m_edge_index.pt')
    total_m2m_edges = sum([e.shape[1] for e in m2m])
    print(f"Graph: {var['name']} | Total M2M Edges: {total_m2m_edges}")

## 6. Visualizing Graph Structure

After creating a graph, it's helpful to visualize its structure to verify the connectivity and understand how grid nodes connect to mesh nodes. Neural-LAM provides a CLI command for creating interactive 3D visualizations of graph structures.

### The `python -m neural_lam.plot_graph` Command

The `python -m neural_lam.plot_graph` command creates an interactive 3D visualization using Plotly. This visualization shows:

- **Grid nodes** (black dots): The original data grid points
- **Mesh nodes** (blue dots): The internal latent mesh nodes
- **G2M edges** (black lines): Grid-to-Mesh connections
- **M2G edges** (black lines): Mesh-to-Grid connections
- **M2M edges** (blue lines): Mesh-to-Mesh connections within each level
- **Hierarchical edges** (green lines): Up/down connections between mesh levels (for hierarchical graphs)

### Command Syntax

```bash
python -m neural_lam.plot_graph \
  --datastore_config_path <path_to_config> \
  --graph <graph_name> \
  [--save <output.html>] \
  [--show_axis]
```

**Required Arguments:**
- `--datastore_config_path`: Path to the datastore configuration YAML file
- `--graph`: Name of the graph to visualize (must exist in `datastore.root_path/graph/<graph_name>/`)

**Optional Arguments:**
- `--save`: Save the visualization as an interactive HTML file instead of displaying it
- `--show_axis`: Display the 3D axes (hidden by default for cleaner visualization)

### Example: Visualizing the Flat Graph

Let's visualize the flat graph we created earlier. Run this command in your terminal:

```bash
# Display interactive plot in browser
python -m neural_lam.plot_graph \
  --datastore_config_path ../data/config.yaml \
  --graph flat_l1

# Or save to HTML file for sharing
python -m neural_lam.plot_graph \
  --datastore_config_path ../data/config.yaml \
  --graph flat_l1 \
  --save flat_graph_visualization.html
```

The visualization will open in your browser, where you can:
- Rotate the view by clicking and dragging
- Zoom in/out with the scroll wheel
- Toggle edge types on/off by clicking legend items
- Hover over nodes to see their positions

### Comparing Flat vs Hierarchical Graphs

The visualization clearly shows the structural differences:

**Flat Graph:**
- All mesh nodes at the same height level
- Node size indicates connectivity (degree)
- Only M2M edges (blue) connect mesh nodes horizontally

**Hierarchical Graph:**
- Mesh nodes arranged in vertical levels
- Each level has its own M2M edges (blue)
- Green edges connect levels (up/down)
- Higher levels have fewer nodes (coarser resolution)


### Understanding the Visualization

The 3D layout helps you verify:

1. **Coverage**: All grid nodes (black) should have G2M connections to nearby mesh nodes
2. **Connectivity**: Mesh nodes should be well-connected via M2M edges
3. **Hierarchy**: For hierarchical graphs, check that levels are properly connected
4. **Density**: Mesh node density should match your design (denser in regions of interest)

If you notice issues like isolated nodes or unexpected connectivity patterns, you may need to adjust graph creation parameters like the number of levels or mesh node placement strategy.